In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

RAW_BASE = ('https://raw.githubusercontent.com/a8nhy6c/hw_bcp_test_files/'
            'main/margin_data/1.2V/')
CSV_URLS = [RAW_BASE + 'hold_margin_1.csv',
            RAW_BASE + 'read_margin_1.csv']

SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
SQUARE_COLOR = '#1baf7a'
GRID = 'rgba(0,0,0,0.08)'
N_GRID = 20001

MIRROR_COLS = {'X', 'Y', 'X_m', 'Y_m'}
SERIES_NAMES = {'Y': 'Original', 'Y_m': 'Mirror'}


def _resample(x, y, grid):
    order = np.argsort(x)
    return np.interp(grid, np.asarray(x)[order], np.asarray(y)[order])


def _largest_square(x, upper, lower):
    """Largest axis-aligned square inside the band between upper and lower.

    Returns (x0, y0, side); the square is centred in whatever slack is left.
    """
    def feasible(s):
        out = []
        for j, a in enumerate(x):
            if a + s > x[-1]:
                break
            k = np.searchsorted(x, a + s)
            hi, lo = upper[j:k + 1].min(), lower[j:k + 1].max()
            if hi - lo >= s:
                out.append((a, lo, hi))
        return out

    lo_s, hi_s = 0.0, float(max(0.0, (upper - lower).max()))
    for _ in range(60):
        s = (lo_s + hi_s) / 2
        if feasible(s):
            lo_s = s
        else:
            hi_s = s

    fits = feasible(lo_s)
    if not fits:
        return None
    a, ylo, yhi = fits[len(fits) // 2]
    return a, ylo + (yhi - ylo - lo_s) / 2, lo_s


def butterfly_squares(ox, oy, mx, my, n=N_GRID):
    """Largest square in each lobe of the original/mirror pair.

    Returns (upper_square, lower_square, crossing_x), squares as (x0, y0, side).
    """
    ox, mx = np.asarray(ox, float), np.asarray(mx, float)
    grid = np.linspace(max(ox.min(), mx.min()), min(ox.max(), mx.max()), n)
    O, M = _resample(ox, oy, grid), _resample(mx, my, grid)

    diff = O - M
    crossings = np.where(np.sign(diff[:-1]) != np.sign(diff[1:]))[0]
    interior = crossings[(grid[crossings] > grid[0] + 1e-6)
                         & (grid[crossings] < grid[-1] - 1e-6)]
    if len(interior) == 0:
        return None, None, None
    x_cross = grid[interior[0]]

    left = grid <= x_cross
    upper = _largest_square(grid[left], O[left], M[left])
    lower = None if upper is None else (upper[1], upper[0], upper[2])
    return upper, lower, x_cross


def series_from(df):
    """Yield (name, x, y) for each line graph held in the dataframe.

    Columns are read in pairs, so both the 2-column input format and the
    4-column X,Y,X_m,Y_m mirrored format work without special-casing names.
    """
    cols = list(df.columns)
    for xcol, ycol in zip(cols[::2], cols[1::2]):
        pair = df[[xcol, ycol]].apply(pd.to_numeric, errors='coerce').dropna()
        if pair.empty:
            continue
        name = SERIES_NAMES.get(ycol, f'{ycol} vs {xcol}')
        yield name, pair[xcol], pair[ycol]


def add_squares(fig, squares):
    for sq in squares:
        if sq is None:
            continue
        x0, y0, s = sq
        fig.add_shape(type='rect', xref='x', yref='y',
                      x0=x0, y0=y0, x1=x0 + s, y1=y0 + s,
                      line=dict(color=SQUARE_COLOR, width=2),
                      fillcolor=SQUARE_COLOR, opacity=0.15, layer='below')
        fig.add_annotation(x=x0 + s / 2, y=y0 + s / 2, text=f'side = {s:.4f}',
                           showarrow=False, font=dict(size=11, color='#333333'))


def plot(url, title=None):
    df = pd.read_csv(url)
    title = title or url.rsplit('/', 1)[-1]
    series = list(series_from(df))
    if not series:
        print(f'Skipping {title}: no numeric column pairs found.')
        return

    fig = go.Figure()
    lo = min(min(x.min(), y.min()) for _, x, y in series)
    hi = max(max(x.max(), y.max()) for _, x, y in series)

    if len(series) > 1:
        fig.add_trace(go.Scatter(
            x=[lo, hi], y=[lo, hi], mode='lines', name='y = x',
            line=dict(color='rgba(0,0,0,0.25)', width=1, dash='dot'),
            hoverinfo='skip',
        ))

    for i, (name, x, y) in enumerate(series):
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines+markers', name=name,
            line=dict(color=SERIES_COLORS[i % len(SERIES_COLORS)], width=2),
            marker=dict(size=5),
            hovertemplate=f'<b>{name}</b><br>x=%{{x:.4g}}<br>y=%{{y:.4g}}<extra></extra>',
        ))

    if MIRROR_COLS <= set(df.columns):
        upper, lower, x_cross = butterfly_squares(
            df['X'], df['Y'], df['X_m'], df['Y_m'])
        if upper is not None:
            print(f'  curves cross at x = {x_cross:.6f} (on y = x)')
            for sq in (upper, lower):
                x0, y0, s = sq
                print(f'  square: side = {s:.6f}, corners '
                      f'({x0:.6f}, {y0:.6f}) to ({x0 + s:.6f}, {y0 + s:.6f})')
            add_squares(fig, [upper, lower])

    fig.update_layout(
        title=title,
        template='plotly_white',
        hovermode='closest',
        showlegend=len(series) > 1,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
        margin=dict(l=60, r=30, t=80, b=60),
        width=760, height=620,
    )
    fig.update_xaxes(title='X', gridcolor=GRID, zeroline=False)
    fig.update_yaxes(title='Y', gridcolor=GRID, zeroline=False,
                     scaleanchor='x', scaleratio=1)
    fig.show()


for url in CSV_URLS:
    print(f'Loading {url}')
    plot(url)

Loading https://raw.githubusercontent.com/a8nhy6c/hw_bcp_test_files/main/margin_data/1.2V/hold_margin_1.csv
  curves cross at x = 0.536954 (on y = x)
  square: side = 0.349372, corners (0.067404, 0.707385) to (0.416777, 1.056758)
  square: side = 0.349372, corners (0.707385, 0.067404) to (1.056758, 0.416777)


Loading https://raw.githubusercontent.com/a8nhy6c/hw_bcp_test_files/main/margin_data/1.2V/read_margin_1.csv
  curves cross at x = 0.595200 (on y = x)
  square: side = 0.156764, corners (0.260022, 0.899982) to (0.416786, 1.056746)
  square: side = 0.156764, corners (0.899982, 0.260022) to (1.056746, 0.416786)
